# ⚾ NRFI/YRFI Model v2
### Split Half-Inning Models + Rolling Pitcher Stats + Leadoff OBP

---

## What changed from v1

| | v1 | v2 |
|---|---|---|
| Pitcher stats | Full season averages (look-ahead bias) | Trailing 10-start rolling (no future data) |
| Batting proxy | Team wOBA (too broad) | Leadoff batter trailing OBP (direct signal) |
| Model structure | Single model, averaged both halves | Two separate models combined via independence formula |

## Combination formula
```
P(YRFI) = 1 - (1 - P_top) x (1 - P_bot)
```
Top and bottom of the 1st are independent events with different pitchers.
Averaging them into one model destroys the signal.

**Runtime:** Uses the same `cache/statcast_2024.csv` — no re-download needed.

## 1 · Install & Imports

In [ ]:
%%capture
!pip install pybaseball pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import os, time, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
import matplotlib.lines as mlines
import seaborn as sns
import pybaseball as pb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score)

pb.cache.enable()
warnings.filterwarnings('ignore')
os.makedirs('cache', exist_ok=True)

ROLLING_N = 10

PARK_FACTORS = {
    'COL':115,'CIN':110,'BOS':108,'PHI':107,'TEX':106,'BAL':104,'NYY':103,
    'CHC':103,'MIL':102,'TOR':101,'ATL':100,'HOU':100,'LAD':100,'WSH':100,
    'CLE':100,'DET':99,'MIN':99,'STL':99,'ARI':99,'MIA':98,'NYM':98,
    'OAK':98,'PIT':98,'SEA':97,'TB':97,'CWS':96,'KC':96,'LAA':96,'SF':95,'SD':94,
}
WIND_DIR_MAP = {
    'Out to CF':1.0,'Out to RF':0.8,'Out to LF':0.8,
    'In from CF':-1.0,'In from LF':-0.8,'In from RF':-0.8,
    'L to R':0.1,'R to L':0.1,'Calm':0.0,'':0.0,
}
TOP_FEATURES = [
    'home_roll_k_pct','home_roll_bb_pct','home_roll_woba','home_roll_dom',
    'away_leadoff_obp','away_team_k_pct','away_team_woba',
    'park_factor','weather_offense_factor',
]
BOT_FEATURES = [
    'away_roll_k_pct','away_roll_bb_pct','away_roll_woba','away_roll_dom',
    'home_leadoff_obp','home_team_k_pct','home_team_woba',
    'park_factor','weather_offense_factor',
]
MONTHLY_RANGES = [
    ('2024-03-20','2024-04-30'),('2024-05-01','2024-05-31'),
    ('2024-06-01','2024-06-30'),('2024-07-01','2024-07-31'),
    ('2024-08-01','2024-08-31'),('2024-09-01','2024-10-01'),
]
print('Setup complete')

## 2 · Load Data
Uses `cache/statcast_2024.csv` if it exists. No re-download needed if you already ran `mlb_models.ipynb`.

In [ ]:
def load_statcast():
    cache = 'cache/statcast_2024.csv'
    if os.path.exists(cache):
        print('Loading from cache...')
        df = pd.read_csv(cache, low_memory=False)
        df['game_date'] = pd.to_datetime(df['game_date'])
        return df
    print('Downloading 2024 Statcast (~20 min)...')
    chunks = []
    for i,(s,e) in enumerate(MONTHLY_RANGES,1):
        print(f'  [{i}/6] {s} to {e}')
        try: chunks.append(pb.statcast(start_dt=s, end_dt=e)); time.sleep(3)
        except Exception as ex: print(f'  Warning: {ex}')
    df = pd.concat(chunks, ignore_index=True)
    df['game_date'] = pd.to_datetime(df['game_date'])
    df.to_csv(cache, index=False)
    return df

sc = load_statcast()
print(f'Loaded {len(sc):,} pitches | {sc["game_pk"].nunique():,} games')
print(f'Date range: {sc["game_date"].min().date()} to {sc["game_date"].max().date()}')

## 3 · Fix 1 — Rolling Pitcher Stats (10-start lookback)

**v1 problem:** Season stats used future games to predict past games.

**v2 fix:** `shift(1).rolling(10)` — only the 10 starts *before* each game are used. `shift(1)` excludes the current game, eliminating all look-ahead bias.

In [ ]:
def build_rolling_pitcher_stats(sc, n=ROLLING_N):
    inn1 = sc[(sc['inning']==1) & sc['events'].notna()].copy()
    gstats = (
        inn1.groupby(['pitcher','game_pk','game_date'])
        .agg(
            pa   = ('events','count'),
            k    = ('events', lambda x:(x=='strikeout').sum()),
            bb   = ('events', lambda x:(x=='walk').sum()),
            woba = ('woba_value','mean'),
        )
        .reset_index()
        .sort_values(['pitcher','game_date'])
    )
    def roll_s(s): return s.shift(1).rolling(n, min_periods=3).sum()
    def roll_m(s): return s.shift(1).rolling(n, min_periods=3).mean()
    g = gstats.groupby('pitcher')
    gstats['roll_pa']   = g['pa'].transform(roll_s)
    gstats['roll_k']    = g['k'].transform(roll_s)
    gstats['roll_bb']   = g['bb'].transform(roll_s)
    gstats['roll_woba'] = g['woba'].transform(roll_m)
    gstats['roll_k_pct']  = gstats['roll_k']  / gstats['roll_pa'].replace(0, np.nan)
    gstats['roll_bb_pct'] = gstats['roll_bb'] / gstats['roll_pa'].replace(0, np.nan)
    gstats['roll_dom']    = gstats['roll_k_pct'] - gstats['roll_bb_pct'] - gstats['roll_woba']
    result = gstats[gstats['roll_pa'] >= 5].copy()
    print(f'Rolling stats: {len(result):,} pitcher-game rows | {result["pitcher"].nunique()} pitchers')
    return result[['pitcher','game_pk','roll_k_pct','roll_bb_pct','roll_woba','roll_dom']]

rolling = build_rolling_pitcher_stats(sc)
rolling.head(3)

## 4 · Fix 2 — Leadoff Batter Trailing OBP

**v1 problem:** Team wOBA averages all 9 batters — drowns the 1st-inning signal.

**v2 fix:** Identify the actual leadoff batter for each half-inning, then compute their trailing OBP from the 50 most recent PA *before* that game.

A leadoff man who gets on base is the direct mechanism for a 1st-inning run.

In [ ]:
def build_leadoff_obp(sc):
    pa = sc[sc['events'].notna()].copy()
    pa['game_date'] = pd.to_datetime(pa['game_date'])
    pa['batting_team'] = np.where(pa['inning_topbot']=='Top', pa['away_team'], pa['home_team'])
    ON_BASE = {'single','double','triple','home_run','walk','hit_by_pitch'}
    pa['on_base'] = pa['events'].isin(ON_BASE).astype(int)

    inn1_pa = pa[pa['inning']==1].sort_values(['game_pk','inning_topbot','at_bat_number','pitch_number'])
    leadoff = (
        inn1_pa.groupby(['game_pk','inning_topbot'])
        .first().reset_index()
        [['game_pk','inning_topbot','game_date','batting_team','batter']]
    )
    batter_pa = (
        pa.groupby(['batter','game_pk','game_date'])
        .agg(pa_count=('events','count'), ob_count=('on_base','sum'))
        .reset_index()
        .sort_values(['batter','game_date'])
    )
    batter_pa['trail_pa'] = batter_pa.groupby('batter')['pa_count'].transform(
        lambda x: x.shift(1).rolling(50, min_periods=10).sum())
    batter_pa['trail_ob'] = batter_pa.groupby('batter')['ob_count'].transform(
        lambda x: x.shift(1).rolling(50, min_periods=10).sum())
    batter_pa['trail_obp'] = batter_pa['trail_ob'] / batter_pa['trail_pa'].replace(0, np.nan)

    lo = leadoff.merge(batter_pa[['batter','game_pk','trail_obp']], on=['batter','game_pk'], how='left')
    lo['trail_obp'] = lo['trail_obp'].fillna(0.320)
    top = lo[lo['inning_topbot']=='Top'][['game_pk','trail_obp']].rename(columns={'trail_obp':'away_leadoff_obp'})
    bot = lo[lo['inning_topbot']=='Bot'][['game_pk','trail_obp']].rename(columns={'trail_obp':'home_leadoff_obp'})
    result = top.merge(bot, on='game_pk', how='outer').fillna(0.320)
    print(f'Leadoff OBP: {len(result):,} games')
    return result

leadoff = build_leadoff_obp(sc)
print(f'Away leadoff OBP mean: {leadoff["away_leadoff_obp"].mean():.3f}')

## 5 · Game Labels + Assemble Dataset

In [ ]:
def build_half_inning_labels(sc):
    inn1 = sc[sc['inning']==1].copy()
    meta = sc.groupby('game_pk').agg(
        game_date=('game_date','first'),home_team=('home_team','first'),away_team=('away_team','first')
    ).reset_index()
    def half_runs(g): return (g['post_bat_score']-g['bat_score']).clip(lower=0).sum()
    tr = inn1[inn1['inning_topbot']=='Top'].groupby('game_pk').apply(half_runs).reset_index(name='top_runs')
    br = inn1[inn1['inning_topbot']=='Bot'].groupby('game_pk').apply(half_runs).reset_index(name='bot_runs')
    meta = meta.merge(tr,on='game_pk',how='left').merge(br,on='game_pk',how='left')
    meta[['top_runs','bot_runs']] = meta[['top_runs','bot_runs']].fillna(0)
    meta['TOP_RUN'] = (meta['top_runs']>0).astype(int)
    meta['BOT_RUN'] = (meta['bot_runs']>0).astype(int)
    meta['YRFI']    = ((meta['TOP_RUN']==1)|(meta['BOT_RUN']==1)).astype(int)
    inn1s = inn1.sort_values(['game_pk','inning_topbot','pitch_number'])
    hp = inn1s[inn1s['inning_topbot']=='Top'].groupby('game_pk')['pitcher'].first().reset_index().rename(columns={'pitcher':'home_starter_id'})
    ap = inn1s[inn1s['inning_topbot']=='Bot'].groupby('game_pk')['pitcher'].first().reset_index().rename(columns={'pitcher':'away_starter_id'})
    meta = meta.merge(hp,on='game_pk',how='left').merge(ap,on='game_pk',how='left')
    for col,default in [('wind_speed',5.0),('wind_dir','Calm'),('temperature',72.0)]:
        if col in sc.columns:
            meta = meta.merge(sc.groupby('game_pk')[col].first().reset_index(),on='game_pk',how='left')
        if col not in meta.columns: meta[col]=default
    meta['wind_speed']  = pd.to_numeric(meta['wind_speed'],errors='coerce').fillna(5.0)
    meta['temperature'] = pd.to_numeric(meta['temperature'],errors='coerce').fillna(72.0)
    meta['wind_dir_factor'] = meta['wind_dir'].astype(str).str.strip().map(WIND_DIR_MAP).fillna(0.0)
    meta['weather_offense_factor'] = (meta['wind_dir_factor']*(meta['wind_speed']/10.0)+(meta['temperature']-72)/30.0)
    meta['park_factor'] = meta['home_team'].map(PARK_FACTORS).fillna(100)/100.0
    meta['game_date'] = pd.to_datetime(meta['game_date'])
    return meta.sort_values('game_date').reset_index(drop=True)

def build_team_stats(sc):
    pa = sc[sc['events'].notna()].copy()
    pa['batting_team'] = np.where(pa['inning_topbot']=='Top', pa['away_team'], pa['home_team'])
    return pa.groupby('batting_team').agg(
        team_woba  =('woba_value','mean'),
        team_k_pct =('events',lambda x:(x=='strikeout').sum()/len(x)),
    ).reset_index()

def assemble_dataset(game_log, rolling, leadoff, team_stats):
    df = game_log.copy()
    hr = rolling.rename(columns={'roll_k_pct':'home_roll_k_pct','roll_bb_pct':'home_roll_bb_pct','roll_woba':'home_roll_woba','roll_dom':'home_roll_dom'})
    ar = rolling.rename(columns={'roll_k_pct':'away_roll_k_pct','roll_bb_pct':'away_roll_bb_pct','roll_woba':'away_roll_woba','roll_dom':'away_roll_dom'})
    df = df.merge(hr[['pitcher','game_pk','home_roll_k_pct','home_roll_bb_pct','home_roll_woba','home_roll_dom']],
                  left_on=['home_starter_id','game_pk'],right_on=['pitcher','game_pk'],how='inner').drop(columns=['pitcher'])
    df = df.merge(ar[['pitcher','game_pk','away_roll_k_pct','away_roll_bb_pct','away_roll_woba','away_roll_dom']],
                  left_on=['away_starter_id','game_pk'],right_on=['pitcher','game_pk'],how='inner').drop(columns=['pitcher'])
    df = df.merge(leadoff,on='game_pk',how='left').fillna({'away_leadoff_obp':0.320,'home_leadoff_obp':0.320})
    ts  = team_stats.rename(columns={'batting_team':'t','team_woba':'away_team_woba','team_k_pct':'away_team_k_pct'})
    ts2 = team_stats.rename(columns={'batting_team':'t','team_woba':'home_team_woba','team_k_pct':'home_team_k_pct'})
    df = df.merge(ts, left_on='away_team',right_on='t',how='left').drop(columns=['t'])
    df = df.merge(ts2,left_on='home_team',right_on='t',how='left').drop(columns=['t'])
    return df.sort_values('game_date').reset_index(drop=True)

print('Building features...')
game_log   = build_half_inning_labels(sc)
team_stats = build_team_stats(sc)
df = assemble_dataset(game_log, rolling, leadoff, team_stats)
df = df.dropna(subset=TOP_FEATURES+BOT_FEATURES+['TOP_RUN','BOT_RUN'])
print(f'Ready: {len(df):,} games | YRFI rate: {df["YRFI"].mean():.1%} | TOP_RUN: {df["TOP_RUN"].mean():.1%} | BOT_RUN: {df["BOT_RUN"].mean():.1%}')

## 6 · Fix 3 — Split Half-Inning Backtest (60/40 time split)

Model A predicts top of 1st. Model B predicts bottom of 1st.
Decision threshold is tuned on the training set rather than hardcoded at 0.5.

In [ ]:
df_s = df.sort_values('game_date').reset_index(drop=True)
split = int(len(df_s)*0.60)
train, test = df_s.iloc[:split], df_s.iloc[split:]
print(f'Train: {len(train):,}  ({train["game_date"].min().date()} to {train["game_date"].max().date()})')
print(f'Test:  {len(test):,}   ({test["game_date"].min().date()} to {test["game_date"].max().date()})')
print(f'Train YRFI: {train["YRFI"].mean():.1%}  |  Test YRFI: {test["YRFI"].mean():.1%}')

sn_top = StandardScaler()
m_top  = LogisticRegression(C=0.3, max_iter=2000, class_weight='balanced', random_state=42)
m_top.fit(sn_top.fit_transform(train[TOP_FEATURES]), train['TOP_RUN'])
p_top_tr = m_top.predict_proba(sn_top.transform(train[TOP_FEATURES]))[:,1]
p_top_te = m_top.predict_proba(sn_top.transform(test[TOP_FEATURES]))[:,1]
print(f'Model A (top 1st): {accuracy_score(test["TOP_RUN"],(p_top_te>=0.5).astype(int)):.1%}')

sn_bot = StandardScaler()
m_bot  = LogisticRegression(C=0.3, max_iter=2000, class_weight='balanced', random_state=42)
m_bot.fit(sn_bot.fit_transform(train[BOT_FEATURES]), train['BOT_RUN'])
p_bot_tr = m_bot.predict_proba(sn_bot.transform(train[BOT_FEATURES]))[:,1]
p_bot_te = m_bot.predict_proba(sn_bot.transform(test[BOT_FEATURES]))[:,1]
print(f'Model B (bot 1st): {accuracy_score(test["BOT_RUN"],(p_bot_te>=0.5).astype(int)):.1%}')

p_yrfi_tr = 1 - (1-p_top_tr)*(1-p_bot_tr)
p_yrfi_te = 1 - (1-p_top_te)*(1-p_bot_te)

best_thresh, best_acc = 0.5, 0.0
for t in np.arange(0.35, 0.75, 0.01):
    a = accuracy_score(train['YRFI'], (p_yrfi_tr>=t).astype(int))
    if a > best_acc: best_acc, best_thresh = a, t

preds = (p_yrfi_te >= best_thresh).astype(int)
acc   = accuracy_score(test['YRFI'], preds)
auc   = roc_auc_score(test['YRFI'], p_yrfi_te)

results = test[['game_date','home_team','away_team','YRFI','TOP_RUN','BOT_RUN']].copy()
results['p_top']            = p_top_te
results['p_bot']            = p_bot_te
results['yrfi_probability'] = p_yrfi_te
results['predicted_YRFI']   = preds
results['correct']          = (preds == test['YRFI'].values).astype(int)

fi_top = pd.DataFrame({'feature':TOP_FEATURES,'coef':m_top.coef_[0],'model':'Top 1st'})
fi_bot = pd.DataFrame({'feature':BOT_FEATURES,'coef':m_bot.coef_[0],'model':'Bot 1st'})
fi = pd.concat([fi_top, fi_bot])

print(f'\nOptimal threshold: {best_thresh:.2f}')
print(f'NRFI v2 accuracy : {acc:.1%}   (v1 was 54.8%)')
print(f'ROC-AUC          : {auc:.3f}  (0.5=random, 1.0=perfect)')
print()
print(classification_report(test['YRFI'], preds, target_names=['NRFI','YRFI'], digits=3))

## 7 · Results Dashboard

**Key panel to watch: bottom-right — Accuracy by Confidence Tier.**
If the strong-conviction tiers (leftmost and rightmost bars) both hit 60%+, the model knows when it's right.

In [ ]:
sns.set_theme(style='darkgrid', palette='muted', font_scale=0.95)
fig = plt.figure(figsize=(22,16))
fig.patch.set_facecolor('#0f1117')
fig.suptitle(
    f'NRFI/YRFI Model v2 -- Split Half-Inning + Rolling Stats + Leadoff OBP\n'
    f'Accuracy: {acc:.1%}  |  ROC-AUC: {auc:.3f}  |  Threshold: {best_thresh:.2f}',
    fontsize=15, fontweight='bold', color='white', y=0.98
)
gs = gridspec.GridSpec(3,3,figure=fig,hspace=0.50,wspace=0.38)
B,P,G,R,O,BG,T='#2196F3','#9C27B0','#4CAF50','#F44336','#FF9800','#1a1d27','#e0e0e0'
def sax(ax,title):
    ax.set_facecolor(BG);ax.set_title(title,fontsize=11,fontweight='bold',color=T,pad=8)
    ax.tick_params(colors=T);ax.xaxis.label.set_color(T);ax.yaxis.label.set_color(T)
    [s.set_edgecolor('#444') for s in ax.spines.values()]

ax1=fig.add_subplot(gs[0,:2])
r=results.sort_values('game_date').copy();r['ra']=r['correct'].rolling(30,min_periods=10).mean()
ov=r['correct'].mean()
ax1.fill_between(r['game_date'],r['ra'],alpha=0.2,color=B)
ax1.plot(r['game_date'],r['ra'],color=B,lw=2,label='30-game rolling')
ax1.axhline(ov,color=R,ls='--',lw=1.5,label=f'Overall: {ov:.1%}')
ax1.axhline(0.50,color='gray',ls=':',alpha=0.6,label='50% baseline')
ax1.axhline(0.548,color=O,ls=':',alpha=0.8,label='v1: 54.8%')
ax1.set_ylim(0.35,0.85);ax1.set_ylabel('Accuracy',color=T)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax1.legend(facecolor=BG,labelcolor=T,fontsize=9)
sax(ax1,'NRFI/YRFI v2 -- 30-Game Rolling Accuracy')

ax2=fig.add_subplot(gs[0,2])
ft=fi[fi['model']=='Top 1st'].sort_values('coef',key=abs,ascending=False).head(7).iloc[::-1]
ax2.barh(ft['feature'],ft['coef'],color=[G if c>0 else R for c in ft['coef']],height=0.6)
ax2.axvline(0,color='white',lw=0.8)
sax(ax2,'Top 1st Features\n(+Run Likely  -NRFI)')

ax3=fig.add_subplot(gs[1,0])
cm_=confusion_matrix(results['YRFI'],results['predicted_YRFI'])
sns.heatmap(cm_,annot=True,fmt='d',cmap='Blues',ax=ax3,
            xticklabels=['Pred NRFI','Pred YRFI'],yticklabels=['Act NRFI','Act YRFI'],
            linewidths=0.5,linecolor='#333',annot_kws={'size':13,'weight':'bold'})
ax3.set_facecolor(BG);ax3.tick_params(colors=T)
ax3.set_title('Confusion Matrix v2',fontsize=11,fontweight='bold',color=T,pad=8)

ax4=fig.add_subplot(gs[1,1])
results[results['YRFI']==0]['yrfi_probability'].hist(ax=ax4,bins=25,alpha=0.65,color=B,label='Actual NRFI',density=True)
results[results['YRFI']==1]['yrfi_probability'].hist(ax=ax4,bins=25,alpha=0.65,color=O,label='Actual YRFI',density=True)
ax4.axvline(best_thresh,color='white',ls='--',lw=1.5,label=f'Threshold: {best_thresh:.2f}')
ax4.set_xlabel('Predicted YRFI Probability',color=T)
ax4.legend(facecolor=BG,labelcolor=T,fontsize=9)
sax(ax4,'Probability Separation v2\n(wider gap vs v1 = better model)')

ax5=fig.add_subplot(gs[1,2])
fb=fi[fi['model']=='Bot 1st'].sort_values('coef',key=abs,ascending=False).head(7).iloc[::-1]
ax5.barh(fb['feature'],fb['coef'],color=[G if c>0 else R for c in fb['coef']],height=0.6)
ax5.axvline(0,color='white',lw=0.8)
sax(ax5,'Bot 1st Features\n(+Run Likely  -NRFI)')

ax6=fig.add_subplot(gs[2,0])
colors_pt=[G if y==1 else R for y in results['YRFI']]
ax6.scatter(results['p_top'],results['p_bot'],c=colors_pt,alpha=0.3,s=12)
ax6.axvline(0.5,color='white',ls=':',alpha=0.5);ax6.axhline(0.5,color='white',ls=':',alpha=0.5)
ax6.set_xlabel('P(Run in Top 1st)',color=T);ax6.set_ylabel('P(Run in Bot 1st)',color=T)
ax6.legend(handles=[
    mlines.Line2D([],[],marker='o',color='w',markerfacecolor=G,markersize=8,label='YRFI'),
    mlines.Line2D([],[],marker='o',color='w',markerfacecolor=R,markersize=8,label='NRFI'),
],facecolor=BG,labelcolor=T,fontsize=9)
sax(ax6,'P(Top Run) vs P(Bot Run)\nGreen=YRFI  Red=NRFI')

ax7=fig.add_subplot(gs[2,1])
r2=results.copy();r2['month']=pd.to_datetime(r2['game_date']).dt.strftime('%b')
ORDER=['Mar','Apr','May','Jun','Jul','Aug','Sep','Oct']
mo=r2.groupby('month')['correct'].agg(acc='mean',n='count').reset_index()
mo['month']=pd.Categorical(mo['month'],categories=ORDER,ordered=True)
mo=mo.sort_values('month').dropna(subset=['month'])
bars=ax7.bar(mo['month'],mo['acc']*100,color=[G if v>55 else O if v>50 else R for v in mo['acc']],edgecolor='none')
ax7.axhline(50,color='gray',ls=':',alpha=0.6)
ax7.axhline(54.8,color=O,ls='--',alpha=0.8,lw=1.5,label='v1: 54.8%')
ax7.set_ylim(40,80);ax7.set_ylabel('Accuracy %',color=T)
for bar,row in zip(bars,mo.itertuples()):
    ax7.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.5,f'{row.acc*100:.0f}%',ha='center',va='bottom',color=T,fontsize=9)
ax7.legend(facecolor=BG,labelcolor=T,fontsize=9)
sax(ax7,'v2 Accuracy by Month\n(Green>55  Orange>50  Red<50)')

ax8=fig.add_subplot(gs[2,2])
bins_=[0,0.35,0.45,0.55,0.65,1.01]
lbls=['<35pct\nStrong NRFI','35-45pct\nLean NRFI','45-55pct\nToss-up','55-65pct\nLean YRFI','>65pct\nStrong YRFI']
results['tier']=pd.cut(results['yrfi_probability'],bins=bins_,labels=lbls)
ta=results.groupby('tier',observed=True)['correct'].agg(acc='mean',n='count').reset_index()
bars2=ax8.bar(ta['tier'],ta['acc']*100,color=[G if v>55 else O if v>50 else R for v in ta['acc']],edgecolor='none')
for bar,row in zip(bars2,ta.itertuples()):
    ax8.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.5,
             f'{row.acc*100:.0f}%\nn={row.n}',ha='center',va='bottom',color=T,fontsize=8)
ax8.axhline(50,color='gray',ls=':',alpha=0.6)
ax8.set_ylim(35,80);ax8.set_ylabel('Accuracy %',color=T)
sax(ax8,'Accuracy by Confidence Tier\n(Does model know when it is right?)')

plt.savefig('nrfi_model_v2_results.png',dpi=150,bbox_inches='tight',facecolor=fig.get_facecolor())
plt.show()
print('Chart saved: nrfi_model_v2_results.png')

## 8 · Monthly Breakdown + Save Results

In [ ]:
r2=results.copy();r2['month']=pd.to_datetime(r2['game_date']).dt.strftime('%b')
monthly=r2.groupby('month')['correct'].agg(acc='mean',n='count')
print(f"{'Month':<8}{'Acc':>8}{'N':>6}")
print('-'*24)
for m,row in monthly.iterrows():
    print(f"{m:<8}{row['acc']*100:>6.1f}%{int(row['n']):>6}  {'X'*int(row['acc']*100/5)}")
print(f'\nv1 overall: 54.8%')
print(f'v2 overall: {acc:.1%}  {"IMPROVEMENT" if acc>0.548 else "see notes"}')
print(f'ROC-AUC:    {auc:.3f}')

results.to_csv('nrfi_v2_backtest_2024.csv', index=False)
print('\nSaved: nrfi_v2_backtest_2024.csv')
try:
    from google.colab import files
    files.download('nrfi_v2_backtest_2024.csv')
    files.download('nrfi_model_v2_results.png')
    print('Downloads triggered.')
except ImportError:
    print('(Not in Colab - files saved locally)')